In [3]:
# ============================================================
# 네이버 블로그 검색 결과 크롤러 (무한스크롤 포함)
# ============================================================

# 1. 필요한 모듈 로딩
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
from bs4 import BeautifulSoup
import pandas as pd
import time


# 2. 사용자 입력
print("=" * 100)
print("네이버 블로그 자료 수집용 웹크롤러입니다.")
print("=" * 100)

query_txt = input("1. 수집할 키워드를 입력하세요: ")
collect_count = int(input("2. 몇 개의 게시물을 수집하시겠습니까?: "))

print("\n크롤링을 시작합니다...\n")


# 3. 크롬 옵션 설정
options = Options()
options.add_argument("--start-maximized")

driver = webdriver.Chrome(options=options)

# 4. 네이버 접속
driver.get("https://www.naver.com/")
time.sleep(2)

# 5. 검색어 입력
search_box = driver.find_element(By.ID, "query")
search_box.click()
search_box.send_keys(query_txt)
search_box.send_keys(Keys.ENTER)

time.sleep(2)

# 6. 블로그 탭 클릭
driver.find_element(By.LINK_TEXT, "블로그").click()
time.sleep(3)


# ============================================================
# 7. 무한 스크롤
# ============================================================

prev_height = driver.execute_script("return document.body.scrollHeight")

while True:
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(2)

    curr_height = driver.execute_script("return document.body.scrollHeight")

    if curr_height == prev_height:
        break

    prev_height = curr_height

print("스크롤 완료\n")


# ============================================================
# 8. 데이터 저장용 리스트 생성
# ============================================================

no_list = []
title_list = []
summary_list = []
date_list = []
nickname_list = []


# ============================================================
# 9. 게시물 카드 단위로 추출
# ============================================================

posts = driver.find_elements(By.CSS_SELECTOR,
"div.sds-comps-vertical-layout.sds-comps-full-layout.YWTMk0ahJUsxq4uCx9gX")

print(f"총 {len(posts)}개의 게시물이 검색되었습니다.\n")


# ============================================================
# 10. 반복문으로 데이터 추출
# ============================================================

for idx, post in enumerate(posts, 1):

    if idx > collect_count:
        break

    try:
        title = post.find_element(By.CLASS_NAME,
        "sds-comps-text-type-headline1").text
    except:
        title = ""

    try:
        summary = post.find_element(By.CSS_SELECTOR,
        "span.sds-comps-text-type-body").text
    except:
        summary = ""

    try:
        date = post.find_element(By.CSS_SELECTOR,
        "span.sds-comps-profile-info-subtext").text
    except:
        date = ""

    try:
        nickname = post.find_element(By.CSS_SELECTOR,
        "a[data-heatmap-target='articleSourceJSX_title'] span").text
    except:
        nickname = ""

    # 리스트 저장
    no_list.append(idx)
    title_list.append(title)
    summary_list.append(summary)
    date_list.append(date)
    nickname_list.append(nickname)

    # 화면 출력
    print(f"번호: {idx}")
    print(f"제목: {title}")
    print(f"요약내용: {summary}")
    print(f"작성일자: {date}")
    print(f"블로그 닉네임: {nickname}")
    print("-" * 100)


# ============================================================
# 11. DataFrame 생성
# ============================================================

df = pd.DataFrame({
    "번호": no_list,
    "제목": title_list,
    "요약내용": summary_list,
    "작성일자": date_list,
    "블로그닉네임": nickname_list
})


# ============================================================
# 12. 파일 저장
# ============================================================

csv_name = query_txt + "_naver_blog.csv"
xlsx_name = query_txt + "_naver_blog.xlsx"
txt_name = query_txt + "_naver_blog.txt"

# CSV 저장
df.to_csv(csv_name, index=False, encoding="utf-8-sig")

# 엑셀 저장
df.to_excel(xlsx_name, index=False, engine="openpyxl")

# TXT 저장
with open(txt_name, "w", encoding="utf-8") as f:
    for i in range(len(df)):
        f.write(f"번호: {df['번호'][i]}\n")
        f.write(f"제목: {df['제목'][i]}\n")
        f.write(f"요약내용: {df['요약내용'][i]}\n")
        f.write(f"작성일자: {df['작성일자'][i]}\n")
        f.write(f"블로그 닉네임: {df['블로그닉네임'][i]}\n")
        f.write("=" * 100 + "\n")


print("\n요청하신 데이터 수집이 완료되었습니다.")
driver.quit()

네이버 블로그 자료 수집용 웹크롤러입니다.

크롤링을 시작합니다...

스크롤 완료

총 0개의 게시물이 검색되었습니다.


요청하신 데이터 수집이 완료되었습니다.
